In [ ]:
from pathlib import Path 
from subprocess import run
import os
import sys

# File Permissions 

All operating systems derived from Unix have a system for specifying the &ldquo;privileges&rdquo; that limit access to any file. The macOS operating system, which is based on BSD Linux, has such a system. Any Linux server you use will have a system that is very similar. 

Windows is not derived from Unix. It uses a different system to specify privileges. Even if your personal computer runs Windows, you will find it helpful to understand the system that Unix pioneered. 

In a previous notebook, you may have created an SSH key-pair that is saved in the `.ssh` folder in your user home. In a Jupyter notebook, if you preface a command with an exclamation point, the notebook will have the shell process the command. 

Bash is the best known shell processor. Most systems derived from Unix support Bash. On macOS, the default shell is zsh. It is derived from Bash and for the purposes at hand, behaves just like Bash. 

If you have installed Git for Windows, you have access to a shell processor called Git Bash. It can simulate many of the commands that are available in the versions of Bash that run on most Linux systems. Unfortunately, the way that Windows handles file permissions is so different that it is not possible to have a single notebook that illustrates mechanics of setting file permissions on both macOS and Windows. Instead there are two notebooks, one for macOS; the other for Windows.  

In [2]:
if sys.platform == 'darwin':
    print("You are working on a computer that runs macOS.\n")
    print("Use this notebook to learn about macOS file permissions.") 
elif sys.platform == "win32":
    print("You are working on a computer that runs Windows.")
    print("Switch to the notebook on Windows file permissions.")
    

You are working on a computer that runs macOS.

Use this notebook to learn about macOS file permissions.


# Changing file permissions on macOS

You may have worked through the [ssh_keys notebook](ssh_keys.ipynb). If so, it created some keys for you in the `.ssh` sub-directory of your user home directory. Here is another copy of that function: 

In [3]:

def create_keypair(key_dir: Path, key_name: str) -> None:
    """A function that creates an ed25519 keypair and saves both keys 
    in the directory key_dir with the stem specified by key_name. The 
    secret key will not be protected by a passphrase. 
    """
    passphrase = ""
    secret_key_path = key_dir / key_name

    if secret_key_file.exists():
        print("\nChange the key_name and try again\n")
        print(f"The file {str(secret_key_path)} already exists.\n")
    else:
        cp = run(
                    [
                        "ssh-keygen", 
                        "-f", 
                        str(secret_key_file), 
                        "-N", 
                        passphrase,
                        "-C",
                        key_name,
                        "-q",
                    ]
                )
        if cp.returncode == 0:
            print(f"Keys were created in {str(key_dir)}.\n")
    return 


The goal here is to play with some keys. Instead of creating them keys in the `.ssh` directory where they could be used by SSH, try creating keys in a new directory called `play_keys` and use `temp` as the value for `key_name`. 

In [4]:
key_name = "temp_key"
play_dir = Path.home() / "play_keys"

play_dir.mkdir(exist_ok=True)

secret_key_file =  play_dir / key_name 

create_keypair(play_dir, key_name)


Change the key_name and try again

The file /Users/pr/play_keys/temp_key already exists.



Recall that by using an exclamation mark as the prefix for a command, you can run shell commands in the code cells of a notebook. 

Let's start by using a Python command to change to the directory `play_dir`. Then let's see what's there. 

In [5]:
os.chdir(play_dir)
!ls -la

total 24
drwxr-xr-x    5 pr  staff   160 Mar 17 00:22 .
drwxr-x---+ 127 pr  staff  4064 Mar 16 23:06 ..
-rwxr-xr--    1 pr  staff    17 Mar 17 00:22 hello_script.sh
-rw-rw----    1 pr  staff   399 Mar 16 23:20 temp_key
-rw-r--r--    1 pr  staff    90 Mar 16 23:20 temp_key.pub


The first two lines of output that you see refer to two directories. The one marked with `..` at the end is the directory that contains `play_dir`. The one with the single `.` is the current directory, `play_dir`. Note that the first letter on each line of text is `d` for directory. 

The next two lines start with a `-`. This tells you that each line refers to a file. The first one is the file with the secret key, `temp_keys`. The second one has the public key, `temp_keys.pub`. 

If you ignore the initial dash, there are 9 characters in these two initial groups:

- `rw-------` for temp_key
- `rw-r--r--` for temp_key.pub

You should interpret the nine characters in these two groups as three groups of three. If we add spaces to emphasize that interpration we have 

- `rw- --- ---` for temp_key
- `rw- r-- r--` for temp_key.pub

The first group tells you about the permissions for the owner of the file. 

- `rw-` means `can read, can write, can't execute`

- `r--` means `can read, can't write, can't execute`

The permissions that `ssh-keygen` gave to these files 

- `rw- --- ---` for `temp_key`

imply that the owner of `temp_key` is the only person who can read and write to this file, the one that stores the secret key. In contrast, the permissions for `temp_key.pub`, 

- `rw- r-- r--` for `temp_key.pub`

imply that the owner of the public key can read from and write to the file. People who are part of the file group can also read it. So can anyone who has an account on the computer. 


The third permission is execute. This makes sense only for a file that has an expression that is executable. Lets make one using a trick from the command line. The `echo` command prints anything that follows as output. 

In [6]:
!echo "echo Hello" 

echo Hello


We can tell the command to sent its output to a file instead:

In [7]:
!echo "echo Hello World" > hello_script.sh

Look in the directory to confirm that this command created a new file. 

In [8]:
!ls -la

total 24
drwxr-xr-x    5 pr  staff   160 Mar 17 00:22 .
drwxr-x---+ 127 pr  staff  4064 Mar 16 23:06 ..
-rwxr-xr--    1 pr  staff    17 Mar 17 00:31 hello_script.sh
-rw-rw----    1 pr  staff   399 Mar 16 23:20 temp_key
-rw-r--r--    1 pr  staff    90 Mar 16 23:20 temp_key.pub


Now let's give everyone execute privileges for this file: 

In [9]:
!chmod +x hello_script.sh
!ls -la 

total 24
drwxr-xr-x    5 pr  staff   160 Mar 17 00:22 .
drwxr-x---+ 127 pr  staff  4064 Mar 16 23:06 ..
-rwxr-xr-x    1 pr  staff    17 Mar 17 00:31 hello_script.sh
-rw-rw----    1 pr  staff   399 Mar 16 23:20 temp_key
-rw-r--r--    1 pr  staff    90 Mar 16 23:20 temp_key.pub


Now, the owner, the group members, and all the others with accounts on the computer have an `x` in their privileges triple. All of them can execute this program. 

You are the owner because you created it. Try to run it by giving the shell the full path to this new file, `./hello_script.sh` (Remember that `.` means this directory.)

In [10]:
!./hello_script.sh

Hello World


OK, maybe we don't want all the others to have the ability to execute this file. Run this command and notice that `o` stands for others:

In [11]:
!chmod o-x hello_script.sh
!ls -la

total 24
drwxr-xr-x    5 pr  staff   160 Mar 17 00:22 .
drwxr-x---+ 127 pr  staff  4064 Mar 16 23:06 ..
-rwxr-xr--    1 pr  staff    17 Mar 17 00:31 hello_script.sh
-rw-rw----    1 pr  staff   399 Mar 16 23:20 temp_key
-rw-r--r--    1 pr  staff    90 Mar 16 23:20 temp_key.pub


If `o` stands for others, `g` must be group and `u` must be user. For the file `temp_key`, how would you add read and write privileges for the group: 

In [12]:
!chmod g+r,g+w temp_key 
!ls -la

total 24
drwxr-xr-x    5 pr  staff   160 Mar 17 00:22 .
drwxr-x---+ 127 pr  staff  4064 Mar 16 23:06 ..
-rwxr-xr--    1 pr  staff    17 Mar 17 00:31 hello_script.sh
-rw-rw----    1 pr  staff   399 Mar 16 23:20 temp_key
-rw-r--r--    1 pr  staff    90 Mar 16 23:20 temp_key.pub


How do we know who the owner is? The user name comes in the block of text that comes just after the column of numbers. strings. After that, you'll see that `staff` is the group for this file. 

If you want to see what groups you are in, run the next cell: 

In [13]:
!id

uid=501(pr) gid=20(staff) groups=20(staff),12(everyone),61(localaccounts),79(_appserverusr),80(admin),81(_appserveradm),98(_lpadmin),713(com.apple.sharepoint.group.13),707(com.apple.sharepoint.group.7),708(com.apple.sharepoint.group.8),709(com.apple.sharepoint.group.9),702(com.apple.sharepoint.group.2),711(com.apple.sharepoint.group.11),712(com.apple.sharepoint.group.12),704(com.apple.sharepoint.group.4),710(com.apple.sharepoint.group.10),706(com.apple.sharepoint.group.6),33(_appstore),100(_lpoperator),204(_developer),250(_analyticsusers),395(com.apple.access_ftp),398(com.apple.access_screensharing),399(com.apple.access_ssh),400(com.apple.access_remote_ae),701(com.apple.sharepoint.group.1),703(com.apple.sharepoint.group.3),705(com.apple.sharepoint.group.5)


That's not very readable but you will see that you have a user id `uid` and belong to many groups including `staff`. 

The solution here is to set privileges on the secret key that prevent the user from reading the key. Malware that has the same privileges as the user is also unable to read the secret key. 

If the user directly runs a git command, git will have the privileges of the user and it too will be unable to read the secret key. To run a git command, the user can use `sudo` which means `substitute user` do. The default is for the substitute user to be root, the one user with privileges that are higher than those of any administrator. 

You created an SSH key-pair in a previous notebook. We can use the same function to create a new SSH key-pair that you can use to experiment with privileges. 

Start as before by specifying a value for `key_name`:

Once the file that contains the key exists, the user can use the command  

`sudo chown root:wheel key_name` 

to change the ownership of the file with the secret key so that root is the owner. Every file specifies three types of privileges: for its owner, for its group, and for all users on the computer. 



### 5. On macOS, the default lets malware use keys even if it can read them

### 6. On Windows, malware can read the passphrase

### 7. On macOS, there is a better solution 

### 8. On Windows, there is no way to stop malware from using SSH 

### 4. Requiring Touch ID on macOS 

In this section and the next, I'll show how to set up SSH on macOS so that a Touch is required before it can use your secret key to make an SSH connection to another server. There are three steps. 

#### Step A  

Have macOS prompt you for a Touch ID approval when you run any command using sudo.

#### Step B

Make root the owner of your secret key. 

#### Step C

Make sure that the commands you want to run such as `git push`, `git pull`, and `git clone` run under sudo. 

### How to complete these steps 

The notebook that you used to create your SSH keys, ran Python commands that did everything for you. 

To complete steps A, B and C, it is better if you use the JupyterLab editor to create some files and then run some commands in the macOS terminal. The process will be more secure this way. You'll also end up with a better understanding of the changes you have made. 

Along the way, I'll explain why it is safer for not to use the macOS terminal instead of the JupyterLab terminal. But don't worry. Step A turns out to have many security benefits. Once you have completed it, you can go back to using the JupyterLab terminal. 

To maintain the flow in this notebook, I'll record some videos that illustrate what you should do in the JupyterLab editor and the macOS terminal. You'll have to take these steps yourself. The code in the notebook won't do all the work for you. 

### Step A: Using Touch ID to approve `sudo` requests

You will configure your computer so that any sudo request leads to a prompt that can be approved using Touch ID. This will have no value if you have not set up Touch ID. If you haven't saved at least one fingerprint, follow the instructions and do this now. I recommend saving two: your right index finger and a backup finger or thumb. 

What's the trade-off here? Having two finger prints makes it more likely that one will work. It also increases slightly the odds that someone can log with a fake finger print without being caught. So make one backup but not more. 



Next, you want to change a file that exists at the location `/etc/pam.d/sudo_local`  on up-to-date versions of macOS. If you are not up-to-date, you are on your own. I can't help you. 

The first thing you should do is run the command in this notebook that displays the contents of the template file in this same directory:

In [14]:
!cat /etc/pam.d/sudo_local.template

# sudo_local: local config file which survives system update and is included for sudo
# uncomment following line to enable Touch ID for sudo
#auth       sufficient     pam_tid.so


You want to create a file that follows the instructions by deleting the `#` character at the beginning of the third line and save it locally. Then from the terminal, you should copy the modified file to the location `/etc/pam.d/sudo_local`. 

If you know how to use other editors, there are other ways to achieve the desired result. If you are confident about what you are doing, by all means do this another way. 

### Permissions and the `sudo` command 

Video 

- Open a macOS terminal 
- Make sure that Secure Keyboard Entry has a check showing that it is active
- Enter `ls -la`
- Now enter the command `echo "hello world" > hello.txt`
- Enter `ls la` again 
- Then `cat hello.txt`
- Go to the JupyterLab editor
- Try to read this file
- Add a new line of text 

Explain. What does the left column mean? Use a still image that I can mark up. 

- Enter `ls -la` again

Notice that the owner of the new file is `root`.  
- sudo chown root:wheel hello.txt
- you have to supply your login password to get permission to change ownership to root
- sudo chmod 700 hello.txt
- Enter `ls -la` 

Now we explain what Secure Keyboard Entry is doing.

We can also warn about something that we will have to change. permissions for sudo last for several minutes

If malware has the same permissions as you, it could run a command before the timeout and have sudo permissions. So the next step will be to set the timeout to 0. But first, we need to get Touch working as a way to approve `sudo` requests. 



The video below shows the steps needed to create a new file and the effects of file permissions: 

- use the JupyterLab File Browser to change to the Pubs directory
- open an instance of JupyterLab editor
- try to read hello.txt; can't even read it 
- add some text; try to save; can't
- change to the Files directory; now you can `save as` in Files
- use the Save As option to save the empty file in `Files/sudo_local`
- switch back to this notebook; copy the three lines of sudo_local.template that are displayed above
- switch back to the editor and paste those three lines into the empty document
- delete the `#` at the beginning of the third line to uncomment that line
- save the text file again using its current name and location, `Files/sudo_local`

Be careful: sudo_local is the name of a file, not a command. 

The next video shows how to copy the file from its current location to the place where macOS will look for it:
- open a macOS terminal; change to the location `/etc/pam.d`
- run this command: `sudo cp ~/Gennaker/Channels/DSD/Files/sudo_local sudo_local`
- when prompted, enter your login password; as a security measure, no characters will show as you type in your password
- enter `ls -la` to show the contents of `/etc/pam.d` and confirm that the new file `sudo_local` is there
- notice its new owner and permissions; they mean that only the root user can write to this file


, 

The asymmetry is 

The adjective If you want, you can think of a secret key as a very long password. There are many different ways to think of  

has 

Modern implementations of SSH use passwords that in effect are assembled from more than 40 randomly selected characters, so it is challenge to manage them. When secrets get this long, most people call them keys instead of passwords, so SSH has a key management problem. Whatever you call it, much of the setup work for SSH is devoted to systems that protect the files where the keys are stored but still let SSH get access to these keys when it needs them without overly inconveniencing the person using the computer. 


## The Innovative Part of SSH 
If you have a secret that is hard to guess, your account can still be can still be compromised if you have to reveal the secret to others. The clever thing that asymmetric cryptography does is give you a way to prove that you know a secret without ever revealing it to anyone. It does this by relying functions that have two seemingly inconsistent characteristics. 


### One-Way Function 
It simplifies the discussion to restrict attention to functions $f(\cdot)$ that map a set of individual integers $\{0, 1, ..., \ell\}$ into pairs of integers $$\{(i, j) \text{ for } i, j \in \{0, 1, ..., L\}.$$ To emphasize the difference between a single integer and a pair, use lower case letters for the singletons and upper case letters for a pair. Then we can write $$f(m) = M$$ $$f(n) = N$$. 

characters long. 

Much of the work associated with SSH keys is devoted to protecting the keys that it stores in files on your computer. 
If you send any password over the internet to a remote server, you 

- Lot of keys 

- a one way function that preserves addition 


integers $f: \{0, \ell \} \rightarrow C$ 
one way function if n -> f(n)=c is easy but c -> n such that f(n) = c is very hard. 

elements of C is an additive group f(n) + f(m) = f(n+m)

suppose that you have elements N, M in C. Is there a formula for c + d that doesn't require knowledge of the m and n 

if there is, here is how to prove you know a secret s: 

calculate P = f(s) publish

When you contact the server, you pick some number r and calculate R = f(r).

server sends you a challenge c and says: 

Find a value z such that f(z) = R + c @ P. 



if there is a 
example of a long key 

save it in a file? 

leaves you at risk if malware runs on your computer

### A One-Way Function that Preserves Addition

f 

basic idea 

## The Challenge-Response Protocol

a. Create a keypair

b. To register with a server, save your public key there

c. To connect again, send your public key and ask for a connection 

d. The server sends a challenge

e. Sign the challenge and return it

f. The server verifies your signature against the saved copy of your public key

## The Key Management Problem

## The SSH Implementation of the Challenge-Response Protocol

In [15]:
git clone git@github.com:paulmromer/public_repo.git

SyntaxError: invalid syntax (438836536.py, line 1)

In [ ]:
git clone git@github.com:paulmromer/public_repo.git

In [ ]:
ssh-keygen(github)

In [ ]:
ssh = Path.home() / ".ssh" 
ssh.exists()

In [ ]:
print((ssh / "config").read_text())

Log into github and save my public key there. 

image

In [ ]:
git clone git@github.com:paulmromer/private_repo.git

In [ ]:
ssh = Path.home() / ".ssh" 

print(ssh / "config") 

In [ ]:
git clone git@github.com:paulmromer/public_repo.git

## The Key Management Problem

### 1. Put the secret key in a file


### 2. Change the permissions on the file

### 3. Use a password to encrypt the file

### 4. Have a program read the secret into memory 

### 5. Store the password for the file in the macOS keychain

In [ ]:
3. Use a password to encrypt the file
4. Have a program that reads the secret key into memory 

- This notebook will walk you through the steps you need to follow configure your computer to use SSH to connect to github.

- The SSH protocol uses public key cryptography to authenticate with a server.

- You have a private key that is stored in your a folder on your computer. You will need to put a public key onto the server at github.com.

- As a first step, create an account on github.com. You'll supply an email address as a user id and a password. To secure your account, you should eventually set up a second factor for authentication.


SSH authentication relies on a challenge response protocol. 

During registration, a subscriber creates a key pair and stores the public key on the server. 

When the server gets a new connection from someone who claims to that claims to re is a subsequent a subsequent the subscriber comes back and claims to  


client creates a key pair for use with this server. The client stores the public key on the server. The server promises to force anyone who claims to know the secret key for that public to prove it: here is a random string that has never 

by sending a challenge that the claimant has to sign. If 

and 

that is be the owner of that public key by  

and  tells the server to 

says that the server can always tell that it is the 

. 


tells the public key with this and gives the public key to the server. 

tells the server that To register, the client tells the server In the registration process, During 

- The client requests a connection.
- The server sends back a random string of bits as a challenge. "If you can sign that 
- 

connection between a client and a server 

When you want to connect to the server: 
- You send a connection request.
- The server sends back a challenge.
- You use the secret key to sign the challenge.
- You send back the signed challenge.
- The server uses the public key you stored there to verify your signature.


SSH is a much more convenient way to connect with github. It is a particularly good way to sync a folder on your machine with a repo on github. 

Here is are the steps to be able to connect quickly and securely with a remote server. 

1. Create a key pair: a private key and a public key.
2. Save the keys in the `.ssh` directory.
3. Change the file permissions to protect the secret key.
4. Use an independent channel to put the public key on the server. 



To do this, let's define a function. Then run this function with the name you want to use. For example, you might use "github" as the name when you run the function. 


In [ ]:
def start_ssh_agent():
    cmd = ['eval "$(ssh-agent -s)"']
    run(cmd, shell=True, capture_output=True)


In [ ]:
def ssh_add_key(key_name):
    """Runs one time after creating the new key pair."""
    s = str(Path.home() / (".ssh/" + key_name))
    if sys.platform == "darwin":
        cp = run(["ssh-add", s], capture_output=True)
        if cp.returncode:
            print(cp.stderr)
    elif sys.platform == "win32":
        s = "c:" + s
        cp = run(["ssh-add", s], capture_output=True)
        if cp.returncode:
            print(cp.stderr)


In [ ]:
def set_file_permissions(f: Path, mode: int):
    """Sets the permissions of a file.

    Args:
      f: The path for a file.
      mode: The permission mode.
    """
    try:
        os.chmod(f, mode)
    except OSError as e:
        print(f"Error setting permissions for {f}: {e}")

In [ ]:
def generate_keys(key_name):
    """Creates a key-pair (key_name, key_name.pub) and puts them in
    your .ssh folder.

    If you already have a keypair with these names, this will overwrite them.

    On macOS, the secret key is protected with a password.
    """
    home_path = Path.home().absolute()
    ssh_dir = home_path / ".ssh"
    if not ssh_dir.exists():
        ssh_dir.mkdir()
    key_path = ssh_dir / key_name

    if key_path.is_file():
        key_path.unlink()

    ls = [
        "ssh-keygen",
        "-q",
        "-t",
        "ed25519",
        "-f",
        str(key_path),
        "-N",
        "",
        "-C",
        key_name,
    ]
    cp = run(ls)
    if key_path.exists():
        set_file_permissions(key_path, 0o600)
    if key_path.with_suffix(".pub").exists():
        set_file_permissions(key_path.with_suffix(".pub"), 0o644)
    if cp.stderr:
        print(cp.stderr)
    return

In [ ]:
def name_config(url, key_name):
    ssh_dir = Path.home() / ".ssh"
    if not ssh_dir.exists():
        ssh_dir.mkdir()
    skey = (ssh_dir / key_name).exists()
    pkey = (ssh_dir / (key_name + ".pub")).exists()
    if skey or pkey:
        print()
        print("In your .ssh folder:")
        if skey:
            print(f"    You already have a secret key named {key_name}")
        if pkey:
            print(f"    You already have a public key named {key_name}" + ".pub")
        print()
        print("Delete the existing ones or try again with a new name.")
        return

    cfg = ssh_dir / "config"
    if not cfg.exists():
        cfg.touch()

    c = cfg.read_text()

    if url in c:
        existing_entry_with_url = True
        print()
        print("You seem to have an entry for this host in your config file.")
    else:
        existing_entry_with_url = False

    if key_name in c:
        print("")
        print("The existing config uses the keypair name you have entered. ")

    if url in c or key_name in c:
        print("Here is your existing config file")
        print()
        print(c)

    new_list = []
    new_list.append(f"Host {url}")
    new_list.append("  AddKeysToAgent yes")
    new_list.append("  IdentityFile " + str(Path.home()) + f"/.ssh/{key_name}")

    print("Here is the config block that will replace any existing one.")
    print()

    print("\n".join(new_list))
    print()

    lines = c.split("\n")
    if existing_entry_with_url:
        for n, s in enumerate(lines):
            if f"{url}" in s:
                n1 = n
                n2 = n1
                # short circuit eval prevents error in next line
                while not (n2 >= len(lines) or lines[n2] == ""):
                    n2 = n2 + 1
                break
        new_lines = lines
        new_lines[n1:n2] = new_list
        if not new_lines[-1] == "":
            new_lines.append("")
    elif not existing_entry_with_url:
        if lines == []:
            new_lines = new_list
        elif lines[-1] == "\n":
            new_lines = lines
            new_lines += new_list
        else:
            new_lines = lines
            new_lines.append("\n")
            new_lines += new_list

    cfg.write_text("\n".join(new_lines))
    generate_keys(key_name)
    start_ssh_agent()
    ssh_add_key(key_name)

    return


In [ ]:
key_name = "my_github_key7"

url = "github.com"

name_config(url, key_name)

In [ ]:
def pk_to_clipboard(key_name):
    """Use copykitten.copy() to put the public key on the clipboard"""
    p = Path.home() / (".ssh/" + key_name + ".pub")
    s = p.read_text()
    copykitten.copy(s)
    return


In [ ]:
pk_to_clipboard(key_name)